re-importing all the dataset, and the functions

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
sys.path.append('../src')
from inference import load_pipeline, explain_applicant, build_query_from_shap, retrieve_policy_context, generate_explanation

df = pd.read_parquet("../data/cleaned_loans.parquet")
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

xgb_model, explainer, embedder, collection = load_pipeline()

/Users/vincent/everything code/python/creditLens/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
sample = X_test.sample(1000, random_state=42)
shap_values = explainer.shap_values(sample)

explanation_data = explain_applicant(5, sample, shap_values, xgb_model)
query = build_query_from_shap(explanation_data)
policy_chunks = retrieve_policy_context(embedder, collection, query)
final_explanation = generate_explanation(explanation_data, policy_chunks)

print(final_explanation)

Here is a clear and specific adverse action explanation, compliant with ECOA/Regulation B:

"We denied your loan application based on our credit assessment. Our analysis showed that you have a relatively high credit risk due to several factors. Specifically, your high interest rate (10.29%) and debt-to-income ratio, combined with your history of having high average current balances, contribute to a higher predicted default probability. Additionally, your verification status as 'Source Verified' and your address being in New York, a high-risk state, also factored into our decision."


In [4]:
import textwrap

print(textwrap.fill(final_explanation, width=80))

Here is a clear and specific adverse action explanation, compliant with
ECOA/Regulation B:  "We denied your loan application based on our credit
assessment. Our analysis showed that you have a relatively high credit risk due
to several factors. Specifically, your high interest rate (10.29%) and debt-to-
income ratio, combined with your history of having high average current
balances, contribute to a higher predicted default probability. Additionally,
your verification status as 'Source Verified' and your address being in New
York, a high-risk state, also factored into our decision."
